In [1]:
import utils
import keras
import numpy as np
from aim.keras import AimCallback
import pandas as pd
from keras import layers

2026-03-13 20:00:44.034853: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-13 20:00:44.046795: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773428444.056271   11387 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773428444.059222   11387 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-13 20:00:44.069236: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
columns = ("lvl1", "lvl2", "lighting")
(train_x, train_y), (val_x, val_y), (test_x, test_y) = utils.read_andre_data()

Correction complete!
 - Files found as-is: 2335
 - Extensions corrected: 1431
 - Still missing (not found): 0
Correction complete!
 - Files found as-is: 497
 - Extensions corrected: 310
 - Still missing (not found): 0
Correction complete!
 - Files found as-is: 476
 - Extensions corrected: 331
 - Still missing (not found): 0


In [3]:
print(train_x.shape)
print(train_y.shape)
print(val_x.shape)
print(val_y.shape)
print(test_x.shape)
print(test_y.shape)

(3766, 300, 300, 3)
(3766, 10)
(807, 300, 300, 3)
(807, 10)
(807, 300, 300, 3)
(807, 10)


In [4]:
input = keras.Input(shape=(train_x[0].shape))

x = keras.layers.Conv2D(filters=32, kernel_size=3, activation="relu")(input)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = keras.layers.MaxPool2D()(x)
x = keras.layers.Flatten()(x)

lvl1 = keras.layers.Dense(1, activation="sigmoid", name="lvl1")(x)
lvl2 = keras.layers.Dense(8, activation="softmax", name="lvl2")(x)

model = keras.Model(inputs=input, outputs={"lvl1": lvl1, "lvl2": lvl2}, name="Overfitted_model")
model.summary()

opt = keras.optimizers.Adam(learning_rate=6e-4)

model.compile(
    optimizer=opt,
    loss={
        "lvl1": keras.losses.BinaryCrossentropy(),
        "lvl2": keras.losses.SparseCategoricalCrossentropy(),
    },
    metrics={
        "lvl1": [keras.metrics.BinaryAccuracy(name="acc")],
    },
    weighted_metrics={
        "lvl2": [keras.metrics.SparseCategoricalAccuracy(name="acc")],
    },
)


I0000 00:00:1773428497.015147   11387 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9165 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "Overfitted_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 300,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 298, 298,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 149, 149,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 147, 147,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 73, 73,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 71, 71,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 35, 35,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 156800)    │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lvl1 (Dense)        │ (None, 1)         │    156,801 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lvl2 (Dense)        │ (None, 8)         │  1,254,408 │ flatten[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,504,457 (5.74 MB)

 Trainable params: 1,504,457 (5.74 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# In case of any issues the following command can be used to restore data:
# aim storage --repo /mnt/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions restore '*'


batch_size = 16
epochs = 10
aim_cb = AimCallback(
    repo=".",                      
    experiment="Vehicle_lvl_Test",
)

hparams = {
    "learning_rate": opt.learning_rate,
    "batch_size": batch_size,
    "epochs": epochs,
    "optimizer": "Adam",
    "model": "Overfitted_model",
}

sample_weight={
    "lvl1": np.ones(len(train_y)),
    "lvl2": train_y['lvl1']
}

validation_data=(val_x, {'lvl1': val_y["lvl1"], 'lvl2': val_y["lvl2"]})

y = {'lvl1': train_y["lvl1"], 'lvl2': train_y["lvl2"]}
history = model.fit(epochs=epochs, batch_size=batch_size, x=train_x, y=y, sample_weight=sample_weight, validation_data=validation_data, callbacks=[aim_cb])

ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
y = {'lvl1': val_y["lvl1"], 'lvl2': val_y["lvl2"]}
model.evaluate(x=val_x, y=y)

26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 11.2620 - lvl1_acc: 0.6803 - lvl1_loss: 1.6932 - lvl2_acc: 0.2045 - lvl2_loss: 9.6485


[11.262015342712402,
 1.6932340860366821,
 9.64846420288086,
 0.6802973747253418,
 0.2044609636068344]

In [ ]:
def predict_heads(model, data_x):
    """Returns (lvl1_probs, lvl2_probs) for the overfitted model."""
    preds = model.predict(data_x, verbose=0)
    # Mapping 'lvl1' -> lvl1 and 'lvl2' -> lvl2
    return preds['lvl1'].flatten(), preds['lvl2']

# Get predictions once to use in all tables
p1, p2 = predict_heads(model, val_x)
# Extract ground truth as numpy arrays
y1_true = val_y['lvl1'].to_numpy()
y2_true = val_y['lvl2'].to_numpy()

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score, f1_score

def eval_lvl1_by_lighting(y_true, p1_probs, lighting_series, threshold=0.5):
    rows = []
    y_pred = (p1_probs >= threshold).astype(int)
    
    for cond in ["Light", "Medium", "Dark"]:
        mask = (lighting_series == cond)
        if not mask.any(): continue
        
        yt, yp = y_true[mask], y_pred[mask]
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        
        rows.append({
            "lighting": cond,
            "n_total": len(yt),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp,
            "TPR": tp / (tp + fn) if (tp + fn) > 0 else 0,
            "TNR": tn / (tn + fp) if (tn + fp) > 0 else 0,
            "lvl1_acc": accuracy_score(yt, yp),
            "lvl1_bal_acc": balanced_accuracy_score(yt, yp),
            "lvl1_f1": f1_score(yt, yp)
        })
    return pd.DataFrame(rows)

print("\n--- TABLE 1: LEVEL 1 BY LIGHTING ---")
display(eval_lvl1_by_lighting(y1_true, p1, val_y['lighting']))


--- TABLE 1: LEVEL 1 BY LIGHTING ---


,lighting,n_total,TN,FP,FN,TP,TPR,TNR,lvl1_acc,lvl1_bal_acc,lvl1_f1
0,Light,442,227,55,71,89,0.556250,0.804965,0.714932,0.680607,0.585526
1,Medium,185,60,33,30,62,0.673913,0.645161,0.659459,0.659537,0.663102
2,Dark,180,47,43,26,64,0.711111,0.522222,0.616667,0.616667,0.649746


In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support
import pandas as pd

TESLA_NAMES = ["S 2012–2015","S 2016–nå", "3 2017–2023", "3 2024–nå","X", "Y 2020–2024", "Y 2025-nå" ]

# 1. Get Predictions
preds = model.predict(val_x, verbose=0)
p1 = preds['lvl1'].flatten() # Level 1 probabilities
p2 = preds['lvl2']           # Level 2 probabilities (softmax)

# 1. Lag en mapping fra navn til indeks
name_to_id = {name: i for i, name in enumerate(TESLA_NAMES)}

# 2. Prepare Ground Truth & Predictions
y1_true = val_y['lvl1'].to_numpy()

# Konverter lvl2-kolonnen til tall (0-6) basert på TESLA_NAMES
y2_true = val_y['lvl2'].map(name_to_id).fillna(-1).to_numpy()

p2 = preds['lvl2']
y2_pred = np.argmax(p2, axis=1)

# Resten av maskeringen din er nå korrekt
tesla_mask = (y1_true == 1)
yt2_tesla = y2_true[tesla_mask]
yp2_tesla = y2_pred[tesla_mask]

# --- TABLE: LEVEL 2 OVERALL ---
def eval_lvl2_overall(yt, yp):
    return pd.DataFrame([{
        "n_tesla": len(yt),
        "lvl2_acc": accuracy_score(yt, yp),
        "lvl2_bal_acc": balanced_accuracy_score(yt, yp),
        "lvl2_f1_macro": f1_score(yt, yp, average='macro'),
        "lvl2_maj_acc": (yt == pd.Series(yt).mode()[0]).mean()
    }])

print("\n--- LEVEL 2 OVERALL ---")
display(eval_lvl2_overall(yt2_tesla, yp2_tesla))

# --- TABLE: LEVEL 2 BY LIGHTING ---
def eval_lvl2_lighting(yt, yp, light):
    rows = []
    for cond in ["Light", "Medium", "Dark"]:
        m = (light == cond)
        if not m.any(): continue
        rows.append({
            "lighting": cond,
            "n_tesla": m.sum(),
            "lvl2_acc": accuracy_score(yt[m], yp[m]),
            "lvl2_bal_acc": balanced_accuracy_score(yt[m], yp[m]),
            "lvl2_f1_macro": f1_score(yt[m], yp[m], average='macro')
        })
    return pd.DataFrame(rows)

print("\n--- LEVEL 2 BY LIGHTING ---")
display(eval_lvl2_lighting(yt2_tesla, yp2_tesla, lighting_tesla))

# --- TABLE: LEVEL 2 PER-CLASS ---
p, r, f, s = precision_recall_fscore_support(yt2_tesla, yp2_tesla, labels=range(len(TESLA_NAMES)), zero_division=0)
class_metrics = pd.DataFrame({
    'class_id': range(len(TESLA_NAMES)),
    'class': TESLA_NAMES,
    'support': s,
    'precision': p,
    'recall': r,
    'f1': f
}).sort_values('recall') # Baseline sorts by recall

print("\n--- LEVEL 2 PER-CLASS ---")
display(class_metrics)

# --- TABLE: TOP MISCLASSIFICATIONS ---
misclassed = []
for t_id in range(len(TESLA_NAMES)):
    for p_id in range(len(TESLA_NAMES)):
        if t_id == p_id: continue
        count = ((yt2_tesla == t_id) & (yp2_tesla == p_id)).sum()
        support = (yt2_tesla == t_id).sum()
        
        if count > 0 and support > 0:
            misclassed.append({
                'true': TESLA_NAMES[t_id],
                'pred': TESLA_NAMES[p_id],
                'rate': count / support,
                'support_true': support
            })

print("\n--- TOP MISCLASSIFICATIONS ---")
if len(misclassed) > 0:
    top_mis = pd.DataFrame(misclassed).sort_values('rate', ascending=False).head(5)
    display(top_mis)
else:
    print("Ingen feilklassifiseringer funnet mellom Tesla-modellene! (Perfekt score eller tomt utvalg)")


--- LEVEL 2 OVERALL ---


/media/d/skole/dat191/visual-vehicle-recognition-varying-lighting-conditions/.venv_l/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,n_tesla,lvl2_acc,lvl2_bal_acc,lvl2_f1_macro,lvl2_maj_acc
0,342,0.0,0.0,0.0,1.0



--- LEVEL 2 BY LIGHTING ---


NameError: name 'lighting_tesla' is not defined